# Démarche de modélisation

**Objectif** : recommander une culture (22 classes) à partir de 7 mesures du sol et du climat.

**Démarche**

1. Données et objectif
2. Qualité des données
3. Analyse exploratoire
4. Protocole d'évaluation
5. Comparaison de modèles
6. Réglage des hyperparamètres
7. Évaluation finale sur le jeu de test
8. Importance des variables
9. Conclusions

Chaque essai est enregistré dans MLflow, expérience `crop-recommendation-exploration`.

**Prérequis** : serveur MLflow démarré (voir le README), noyau `.venv` du projet.

## Environnement

In [ ]:
import os
import warnings
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")  # racine du projet
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
import logging

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from crop_recommendation_tp.config import load_config, load_yaml
from crop_recommendation_tp.validation import validate

logging.getLogger("mlflow").setLevel(logging.WARNING)
pd.set_option("display.precision", 4)

## 1. Données et objectif

| Variable | Signification |
|---|---|
| `N`, `P`, `K` | azote, phosphore, potassium dans le sol |
| `temperature` | température moyenne (°C) |
| `humidity` | humidité relative (%) |
| `ph` | acidité du sol |
| `rainfall` | pluviométrie (mm) |
| `label` | culture adaptée (cible, 22 valeurs) |

**Mesure de succès : F1 macro.** Un score F1 par culture, puis leur moyenne : chaque culture compte autant, y compris les moins fréquentes. L'accuracy est suivie en complément.

In [ ]:
config = load_config()
target = config["dataset"]["target"]
seed = config["split"]["random_seed"]

df = pd.read_csv(config["dataset"]["path"])
print(df.shape)
df.head()

## 2. Qualité des données

In [ ]:
schema = load_yaml(config["dataset"]["schema"])
print("Erreurs de schéma :", validate(df, schema))
print("Valeurs manquantes :", int(df.isna().sum().sum()))
print("Lignes en double :", int(df.duplicated().sum()))

In [ ]:
effectifs = df[target].value_counts()
print(
    f"{effectifs.size} cultures, de {effectifs.min()} à {effectifs.max()} lignes par culture"
)

**Constat**

- Données conformes au schéma, sans valeur manquante ni doublon.
- Classes parfaitement équilibrées : 100 lignes par culture.

**Conséquence** : aucun traitement préalable n'est nécessaire ; l'accuracy et le F1 macro seront proches.

## 3. Analyse exploratoire

In [ ]:
features = [c for c in schema["columns"] if c != target]
df[features].describe().T

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, features, strict=False):
    ax.hist(df[col], bins=30, color="#97BC62", edgecolor="white")
    ax.set_title(col)
axes.flat[-1].axis("off")
fig.suptitle("Distribution des variables")
fig.tight_layout()

**Profil moyen de chaque culture.** Moyenne de chaque variable par culture, centrée-réduite sur l'ensemble des cultures : une couleur marquée signale une culture qui se distingue nettement sur cette variable.

In [ ]:
profil = df.groupby(target)[features].mean()
profil_z = (profil - profil.mean()) / profil.std()

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(profil_z, cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
ax.set_xticks(range(len(features)), features, rotation=45, ha="right")
ax.set_yticks(range(len(profil_z)), profil_z.index)
fig.colorbar(im, label="écart à la moyenne (en écarts-types)")
ax.set_title("Profil moyen par culture")
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
corr = df[features].corr()
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(features)), features, rotation=45, ha="right")
ax.set_yticks(range(len(features)), features)
for i in range(len(features)):
    for j in range(len(features)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im)
ax.set_title("Corrélations entre variables")
fig.tight_layout()

**Constat**

- Distributions à plusieurs pics pour `N`, `P`, `K`, `humidity` et `rainfall` : chaque pic correspond à un groupe de cultures aux besoins voisins (par exemple, `apple` et `grapes` forment le pic de `K` autour de 200).
- Profils très marqués : `apple` et `grapes` (phosphore et potassium élevés), `rice` (pluviométrie élevée), `chickpea` et `kidneybeans` (humidité faible), `papaya` (température élevée).
- Corrélations faibles, sauf entre `P` et `K` (0,74), due aux deux cultures riches en phosphore et en potassium.

**Conséquences**

- Les cultures sont bien séparées dans l'espace des variables : une classification de haute qualité est attendue.
- Les variables ont des échelles très différentes : les modèles fondés sur des distances ou des coefficients exigent une mise à l'échelle.

## 4. Protocole d'évaluation

Le protocole est fixé **avant** toute modélisation :

- **Jeu de test** : 20 % des lignes, découpage stratifié (même proportion de chaque culture), graine 42. Il est mis de côté et ne sert qu'une fois, à l'évaluation finale.
- **Validation croisée** : 5 plis stratifiés, lignes mélangées avec la même graine, sur le jeu d'entraînement uniquement. Toutes les comparaisons et tous les choix se font sur ce score.
- **Mesure** : F1 macro (moyenne et écart-type sur les 5 plis).

Ces réglages proviennent de `config/config.yaml`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df[target],
    test_size=config["split"]["test_size"],
    random_state=seed,
    stratify=df[target],
)
cv = StratifiedKFold(
    n_splits=config["tuning"]["cv_folds"], shuffle=True, random_state=seed
)
print("Entraînement :", X_train.shape, "· Test :", X_test.shape)

In [ ]:
mlflow.set_tracking_uri(config["mlflow"]["tracking_uri"])
mlflow.set_experiment(config["mlflow"]["experiment_name"] + "-exploration")
resultats = []


def evaluer(nom, modele):
    """Évalue un modèle en validation croisée et enregistre le résultat dans MLflow."""
    scores = cross_validate(
        modele, X_train, y_train, cv=cv, scoring=["f1_macro", "accuracy"]
    )
    ligne = {
        "modèle": nom,
        "f1_macro_moyen": scores["test_f1_macro"].mean(),
        "f1_macro_écart_type": scores["test_f1_macro"].std(),
        "accuracy_moyenne": scores["test_accuracy"].mean(),
    }
    with mlflow.start_run(run_name=nom):
        mlflow.set_tag("etape", "comparaison")
        mlflow.log_params(
            {
                "modele": nom,
                **{k: str(v) for k, v in modele.get_params(deep=False).items()},
            }
        )
        mlflow.log_metrics(
            {
                "cv_f1_macro_mean": ligne["f1_macro_moyen"],
                "cv_f1_macro_std": ligne["f1_macro_écart_type"],
                "cv_accuracy_mean": ligne["accuracy_moyenne"],
            }
        )
    resultats.append(ligne)
    return ligne

## 5. Comparaison de modèles

Trois familles de modèles, de complexité croissante :

- **Régression logistique** : modèle linéaire, simple et interprétable.
- **k plus proches voisins** : décision par similarité avec les parcelles connues.
- **Forêt aléatoire** : ensemble d'arbres de décision, non linéaire.

La régression logistique et les k plus proches voisins sont sensibles à l'échelle des variables (`rainfall` varie de 20 à 300, `ph` de 3 à 10) : une mise à l'échelle (`StandardScaler`) les précède, dans un pipeline ajusté à chaque pli.

In [ ]:
candidats = {
    "régression logistique": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000)
    ),
    "k plus proches voisins": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "forêt aléatoire": RandomForestClassifier(random_state=seed),
}
for nom, modele in candidats.items():
    evaluer(nom, modele)

comparaison = (
    pd.DataFrame(resultats)
    .set_index("modèle")
    .sort_values("f1_macro_moyen", ascending=False)
)
comparaison

**Constat**

- La forêt aléatoire obtient le meilleur score (0,9937 ± 0,0061), devant la régression logistique (0,9678) et les k plus proches voisins (0,9650).
- L'écart avec les deux autres modèles, environ 0,03, dépasse nettement l'écart-type entre plis.

**Décision** : forêt aléatoire retenue. Meilleur score, aucune mise à l'échelle nécessaire, importance des variables disponible.

## 6. Réglage des hyperparamètres

Recherche sur grille pour la forêt aléatoire, avec la grille `tuning.param_grid` de la configuration et la même validation croisée.

In [ ]:
recherche = GridSearchCV(
    RandomForestClassifier(random_state=seed),
    param_grid=config["tuning"]["param_grid"],
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
)
recherche.fit(X_train, y_train)

grille = pd.DataFrame(recherche.cv_results_)[
    ["param_n_estimators", "param_max_depth", "mean_test_score", "std_test_score"]
].sort_values("mean_test_score", ascending=False)

with mlflow.start_run(run_name="réglage forêt aléatoire"):
    mlflow.set_tag("etape", "reglage")
    mlflow.log_params({f"best_{k}": v for k, v in recherche.best_params_.items()})
    mlflow.log_metric("best_cv_f1_macro_mean", recherche.best_score_)

print("Meilleurs paramètres :", recherche.best_params_)
grille

**Constat**

- Meilleure combinaison : 200 arbres, profondeur illimitée (0,9943 ± 0,0048).
- Les combinaisons de profondeur 10 ou illimitée sont équivalentes : leurs écarts, inférieurs à 0,002, restent sous l'écart-type entre plis.
- Une profondeur de 5 ou moins dégrade nettement le score.
- Gain modeste par rapport aux paramètres par défaut (0,9937).

**Décision** : `n_estimators=200`, `max_depth=None`. Ces valeurs serviront à l'entraînement du modèle.

## 7. Évaluation finale sur le jeu de test

Le modèle retenu est entraîné sur tout le jeu d'entraînement, puis mesuré **une seule fois** sur le jeu de test.

In [ ]:
modele_final = RandomForestClassifier(**recherche.best_params_, random_state=seed).fit(
    X_train, y_train
)
predictions = modele_final.predict(X_test)
test_metrics = {
    "test_accuracy": accuracy_score(y_test, predictions),
    "test_f1_macro": f1_score(y_test, predictions, average="macro"),
}

fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, xticks_rotation=90, colorbar=False, cmap="Greens", ax=ax
)
ax.set_title("Matrice de confusion (jeu de test)")
fig.tight_layout()

with mlflow.start_run(run_name="modèle final") as run:
    run_final = run.info.run_id
    mlflow.set_tag("etape", "evaluation_finale")
    mlflow.log_params(recherche.best_params_)
    mlflow.log_metrics(test_metrics)
    mlflow.log_text(
        classification_report(y_test, predictions), "classification_report.txt"
    )
    mlflow.log_figure(fig, "confusion_matrix.png")

test_metrics

In [ ]:
erreurs = pd.DataFrame({"réelle": y_test, "prédite": predictions})
erreurs = erreurs[erreurs["réelle"] != erreurs["prédite"]]
print(f"{len(erreurs)} erreurs sur {len(y_test)} parcelles")
erreurs.merge(X_test, left_index=True, right_index=True)

**Constat**

- Jeu de test : accuracy 0,9955 et F1 macro 0,9955, cohérents avec la validation croisée (0,9943) ; aucun signe de sur-apprentissage.
- 2 erreurs sur 440 parcelles, entre cultures aux conditions voisines : `blackgram` prédit `maize` ; `rice` prédit `jute`, deux cultures de milieux humides et pluvieux.

## 8. Importance des variables

Part de chaque variable dans les décisions de la forêt aléatoire (somme égale à 1).

In [ ]:
importances = pd.Series(modele_final.feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
importances.plot.barh(ax=ax, color="#2C5F2D")
ax.set_title("Importance des variables")
fig.tight_layout()

with mlflow.start_run(run_id=run_final):
    mlflow.log_figure(fig, "importance_des_variables.png")

**Constat**

- Variables les plus utilisées : `rainfall` et `humidity`, puis `K` et `P`.
- `temperature` et `ph` contribuent le moins.

**Conséquence** : en production, la surveillance de la dérive portera en priorité sur ces quatre variables.

## 9. Conclusions

**Modèle retenu** : forêt aléatoire, 200 arbres, profondeur illimitée.

**Performance attendue** : F1 macro de l'ordre de 0,99, pour des parcelles comparables aux données d'entraînement.

**Limites**

- Données équilibrées et très séparables : performance à confirmer sur des données réelles.
- Modèle valable dans le domaine couvert par les données (température 9–44 °C, pluviométrie 20–300 mm…). En production, une surveillance de la dérive est nécessaire.

**Industrialisation** : feuille de route des étapes suivantes. Chaque étape du notebook devient un composant de l'application.

| Étape du notebook | Application |
|---|---|
| Qualité des données | `config/schema.yaml`, `validation.py` |
| Protocole d'évaluation | `data.py` (`split_dataset`, `build_cv`), réglages dans `config/config.yaml` |
| Réglage des hyperparamètres | `uv run tune` |
| Entraînement et évaluation finale | `uv run train` |
| Choix du modèle en service | `uv run promote`, registre MLflow |
| Suivi des essais | `tracking.py`, serveur MLflow |
| Vérifications | `tests/`, intégration continue |